In [1]:
import torch
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModel.from_pretrained("gpt2").to(device)

inputs = tokenizer("AI is", return_tensors="pt").to(device)

# Forward pass (no generation capability)
outputs = model(**inputs)

print("Hidden states shape:", outputs.last_hidden_state.shape)

# Try to "generate" (this will fail)
try:
    model.generate(**inputs)
except Exception as e:
    print("\n Generation failed :")
    print(e)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

C:\Users\Piyush\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Piyush\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Hidden states shape: torch.Size([1, 2, 768])

 Generation failed :
'GPT2Model' object has no attribute 'generate'


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)

inputs = tokenizer("AI is", return_tensors="pt").to(device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    temperature=0.7,
    do_sample=True
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


AI is trying to find out what is being done with the data.

When you start to get a sense of the situation, it becomes clear that the government is trying to use data to gain an advantage in the process.

The government is trying


In [6]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)

input_ids = tokenizer("AI is", return_tensors="pt").input_ids.to(device)

past_key_values = None
max_new_tokens = 30

for _ in range(max_new_tokens):
    # Forward pass (with cache)
    outputs = model(
        input_ids=input_ids,
        past_key_values=past_key_values,
        use_cache=True
    )

    logits = outputs.logits
    past_key_values = outputs.past_key_values  # 🔥 cache reuse

    last_token_logits = logits[:, -1, :]

    # Convert to probabilities
    probs = F.softmax(last_token_logits, dim=-1)

    # Greedy pick (same as your original)
    next_token = torch.argmax(probs, dim=-1).unsqueeze(0)

    # Append token
    input_ids = torch.cat([input_ids, next_token], dim=1)

    # Stop if EOS token appears
    if next_token.item() == tokenizer.eos_token_id:
        break

print("Looped output:", tokenizer.decode(input_ids[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Looped output: AI is a new AI is a new AI is a new AI is a new AI is a new AI is a new AI is a new AI is a new


In [3]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, GPT2LMHeadModel

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)

input_ids = tokenizer("AI is", return_tensors="pt").input_ids.to(device)

# Manual generation loop (fine control)
for _ in range(50):
    outputs = model(input_ids)
    
    logits = outputs.logits[:, -1, :]  # last token logits
    
    # Custom sampling (temperature + top-k)
    probs = F.softmax(logits / 0.7, dim=-1)
    top_k = 50
    top_probs, top_indices = torch.topk(probs, top_k)

    next_token = top_indices[0, torch.multinomial(top_probs[0], 1)]
    
    input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)

print(tokenizer.decode(input_ids[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

AI is a new, innovative approach to the development of social networks and the development of new technologies.

In this role, we will create new and innovative solutions to the problems of communication and the development of new technologies.

We will develop new technologies
